# TsekTxt: test the 3 models (Colab, GPU optional)
Runtime > Change runtime type > T4 GPU (faster, not required). Run cells top to bottom.

This is the notebook to use (test the models, score the Taglish set, IG). It never retrains anything. Open it from https://colab.research.google.com/github/hansjio/tsektxt-model-training/blob/master/notebooks/colab_eval.ipynb

In [ ]:
!git clone -q https://github.com/hansjio/tsektxt-model-training.git
%cd tsektxt-model-training
!pip -q install 'transformers>=4.41,<5' scikit-learn pandas pyyaml scipy captum joblib

## 1. Try any text (all 3 models, loaded from Hugging Face)

In [ ]:
!python src/predict.py --text 'Totoo ba na may dayaan sa eleksyon? Share this para malaman ng lahat!'

## 2. Overfitting check: train/val/test accuracy
Needs the splits. This rebuilds them from data/raw (a few minutes, CPU ok).

In [ ]:
!pip -q install ftfy datasets
!python src/clean_data.py --input data/raw/combined_dataset_clean.csv
!python src/make_splits.py
!python src/predict.py --split-gap

## 2b. Domain-shift checks (inference only, optional)
How accuracy changes when internal test articles are cut short, and whether a link matters. Writes `results/domain_shift.json` / `.md`. Never uses the Taglish set for training or tuning.

In [ ]:
!python src/domain_shift_checks.py

## 3. Evaluate on the Taglish test set
Upload `taglish_test.csv` with columns `id,text,label` (1 = credible, 0 = not credible). Never train on it.

In [ ]:
from google.colab import files
import os
up = files.upload()
os.makedirs('data/external', exist_ok=True)
os.replace(list(up)[0], 'data/external/taglish_test.csv')

In [ ]:
for m in ['roberta-tagalog','mbert','xlmr']:
    !python src/evaluate_external.py --model {m} --csv data/external/taglish_test.csv

## 3b. Statistics, comparison and Integrated Gradients on the Taglish set
Runs the TF-IDF baseline on the same file, then 95% bootstrap CIs, McNemar with Holm correction, `results/comparison.md`, and Integrated Gradients (the IG step is the slow one; T4 GPU recommended). Needs the baseline's training splits, so run Section 2 first.

In [ ]:
!python src/baseline.py --external data/external/taglish_test.csv
!python src/external_stats.py
!python src/compare.py

In [ ]:
!python src/interpret.py corpus --csv data/external/taglish_test.csv

## 3c. Results by post length (fixed bins)
After section 3: splits the Taglish results into under 20 / 20-39 / 40-99 / 100+ words. The primary result stays the full set; 100+ is a labelled subset. Writes `results/external_length_bins.json` / `.md`. Also writes `results/external_confidence_check.json` (how sure the models are when wrong).

In [ ]:
!python src/length_bins.py --csv data/external/taglish_test.csv
!python src/confidence_check.py --file external_predictions.csv

## 4. Download results
Zips `results/external_*`, `results/*/external_*` and `results/ig_*.json`, `results/domain_shift.*` (plus `comparison.md`). Copy them back into your repo's `results/` folder.

In [ ]:
!zip -qr external_results.zip results/external_* results/*/external_* results/ig_*.json results/comparison.md results/domain_shift.*
files.download('external_results.zip')